# 📊 Gold Spot
Notebook 02: Modeling

## Preliminaries


In [1]:
import pandas as pd
import numpy as np

# Algoritmos de Modelado No Supervisado y Reducción de Dimensionalidad
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture

# Escalado
from sklearn.preprocessing import StandardScaler

# Métricas de Evaluación y Selección de Hiperparámetros (k clusters / n_components)
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)

from src.config import clean_data_file

In [2]:
# Open data file
df = pd.read_parquet(clean_data_file)
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 188 entries, 2011-02-28 to 2026-09-30
Data columns (total 20 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   real_yield_10y                        188 non-null    float64
 1   vix_index                             188 non-null    float64
 2   oil_ret_1m                            188 non-null    float64
 3   gold_ret_1m                           188 non-null    float64
 4   spy_ret_1m                            188 non-null    float64
 5   dxy_broad_goods_ret_1m                188 non-null    float64
 6   vix_index_ret_1m                      188 non-null    float64
 7   oil_ret_12m                           188 non-null    float64
 8   gold_ret_12m                          188 non-null    float64
 9   spy_ret_12m                           188 non-null    float64
 10  cpi_headline_lag1_ret_12m             188 non-null    float64
 11  

In [3]:
# Escalar características
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# PCA: Reducción de dimensionalidad a 2 factores principales
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"Varianza explicada por componente: {pca.explained_variance_ratio_}")

# K-Means sobre el espacio PCA
kmeans = KMeans(n_clusters=4, init="k-means++", n_init=10, random_state=42)
kmeans_labels = kmeans.fit_predict(X_pca)

# Gaussian Mixture Model
gmm = GaussianMixture(
    n_components=4,
    covariance_type="full",  # 'full' permite covarianzas cruzadas entre variables
    n_init=5,
    random_state=42,
)
gmm.fit(X_pca)
gmm_labels = gmm.predict(X_pca)
gmm_probs = gmm.predict_proba(X_pca)  # Probabilidad de pertenecer a cada régimen

# Evaluación de la calidad del agrupamiento
print(f"Silhouette Score (K-Means): {silhouette_score(X_pca, kmeans_labels):.4f}")
print(f"Silhouette Score (GMM):     {silhouette_score(X_pca, gmm_labels):.4f}")
print(f"GMM BIC Score:              {gmm.bic(X_pca):.2f}")  # Menor BIC = mejor ajuste

Varianza explicada por componente: [0.19498303 0.1730224 ]
Silhouette Score (K-Means): 0.3603
Silhouette Score (GMM):     0.3540
GMM BIC Score:              1528.71


In [4]:
# Inspeccionar las cargas de los componentes
loadings = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2'],
    index=df.columns
)
print("Cargas factoriales de PCA:")
print(loadings.round(3))

Cargas factoriales de PCA:
                                        PC1    PC2
real_yield_10y                       -0.080 -0.182
vix_index                            -0.137  0.385
oil_ret_1m                            0.258 -0.055
gold_ret_1m                           0.149 -0.022
spy_ret_1m                            0.350 -0.264
dxy_broad_goods_ret_1m               -0.291  0.181
vix_index_ret_1m                     -0.250  0.267
oil_ret_12m                           0.004 -0.187
gold_ret_12m                          0.174  0.134
spy_ret_12m                           0.204 -0.147
cpi_headline_lag1_ret_12m            -0.233 -0.177
industrial_production_lag1_ret_12m   -0.214 -0.293
m2_money_supply_lag1_ret_12m          0.198  0.261
real_yield_10y_change_1m             -0.212  0.024
st_louis_financial_stress_change_1m  -0.307  0.193
unemployment_rate_lag1_change_12m     0.277  0.274
fed_funds_rate_lag1_change_12m       -0.269 -0.290
fed_funds_rate_lag1_change_3m        -0.314 -0.225
unem

In [5]:
# Medias por cluster
df_analysis = df.copy()
df_analysis['regime'] = gmm_labels

profile = df_analysis.groupby('regime').mean().T
print("\nPerfil macro por Régimen:")
print(profile.round(3))


Perfil macro por Régimen:
regime                                     0       1       2       3
real_yield_10y                         0.462  -0.834   0.777   0.755
vix_index                             18.640  28.337  28.879  15.060
oil_ret_1m                            -0.016   0.129  -0.095   0.025
gold_ret_1m                           -0.000   0.009  -0.016   0.016
spy_ret_1m                            -0.009   0.037  -0.067   0.032
dxy_broad_goods_ret_1m                 0.009  -0.007   0.023  -0.005
vix_index_ret_1m                       0.124  -0.016   0.429  -0.097
oil_ret_12m                           -0.014  -0.227  -0.035   0.150
gold_ret_12m                           0.110   0.262   0.042   0.083
spy_ret_12m                            0.111   0.126  -0.039   0.170
cpi_headline_lag1_ret_12m              0.023   0.010   0.048   0.028
industrial_production_lag1_ret_12m     0.007  -0.076   0.010   0.017
m2_money_supply_lag1_ret_12m           0.064   0.218   0.040   0.052
real_yi